In [13]:
!pip install -q transformers datasets peft accelerate

In [14]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch

In [15]:
# [3] Model & tokenizer selection
model_name = "google/flan-t5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)    # loads pretrained weights
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [17]:
# [4] Load a small subset of a dataset for demo (replace with your data)
raw_ds = load_dataset("cnn_dailymail", "3.0.0", split="train[:1%]")

In [18]:
# [5] Preprocessing function: tokenize inputs and targets, mask padding tokens in labels
max_input_length = 512
max_target_length = 128

In [19]:
def preprocess(batch):
    inputs = batch["article"]
    targets = batch["highlights"]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    # Replace pad token id's in labels by -100 so they are ignored by the loss
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in example]
        for example in labels_ids
    ]
    model_inputs["labels"] = labels_ids
    return model_inputs

In [20]:
train_ds = raw_ds.map(preprocess, batched=True, remove_columns=raw_ds.column_names)

Map:   0%|          | 0/2871 [00:00<?, ? examples/s]

In [21]:
# [6] Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # seq2seq tasks
    r=8,                              # low-rank dimension
    lora_alpha=32,                    # scaling
    lora_dropout=0.1,                 # regularization
)

In [22]:
# [7] Apply LoRA adapters to the model (this wraps the model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # print number of trainable params (sanity check)

# [8] Data collator for dynamic padding (important for seq2seq)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [23]:
# [9] Training arguments (tweak for your GPU/requirements)
training_args = TrainingArguments(
    output_dir="./lora_t5",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = batch_size * grad_accum
    num_train_epochs=1,
    learning_rate=1e-4,
    fp16=torch.cuda.is_available(),  # use mixed precision if CUDA is available
    logging_steps=100,
    save_strategy="epoch",
)

In [24]:
# [10] Trainer instantiation
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipython-input-743443188.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
# [11] Train (only LoRA adapter parameters will update)
trainer.train()

# [12] Save adapters (lightweight)
model.save_pretrained("lora_flan_t5")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 